# Pyspark SCD 2 type :-
##      1) Using Hash function to check

In [0]:
# Generating Sample Data Sample initial data
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("PysparkSCD2").getOrCreate()
initial_df = spark.createDataFrame(
    [(1, "Widget A", 10.00),
     (2, "Widget B", 12.50),
     (3, "Widget C", 20.00)],
    ["id", "name", "price"]
)
initial_df.show()
initial_df.printSchema()

+---+--------+-----+
| id|    name|price|
+---+--------+-----+
|  1|Widget A| 10.0|
|  2|Widget B| 12.5|
|  3|Widget C| 20.0|
+---+--------+-----+

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- price: double (nullable = true)



**Adding a column for merge condition,That will be key column to insert or update
**

In [0]:
from pyspark.sql.functions import sha2,lit,concat_ws,col
initial_df=initial_df.withColumn("new_hash",sha2(concat_ws(lit('_'),col("id"),col("name")),256))
initial_df.show(truncate=False)

+---+--------+-----+----------------------------------------------------------------+
|id |name    |price|new_hash                                                        |
+---+--------+-----+----------------------------------------------------------------+
|1  |Widget A|10.0 |bc4ca3e7265ea6648b12cb219796a29fb57490b724e2e35573c58ecf3e2a8966|
|2  |Widget B|12.5 |b2f924f33af193f4bdd5732036f7e911556c30dceb3a5f5527e859de7b416e84|
|3  |Widget C|20.0 |021b016051b782d0982dec293c01002a8e94d42a9ff1fb683932ce3277b86d7b|
+---+--------+-----+----------------------------------------------------------------+



In [0]:
# Importing current_timestamp package 
from pyspark.sql.functions import current_timestamp

In [0]:
#Dropping table if exists
spark.sql("Drop table main.demo_schema.scd2_table_demo")

DataFrame[]

In [0]:
# 
spark.sql("""
          Create table main.demo_schema.scd2_table_demo 
           (
          new_hash string  NOT NULL PRIMARY KEY,
          id int,
          name string,
          price double,
          effective_date timestamp,
          end_date timestamp
          ) """)

DataFrame[]

Others ways to create a delta table from dataframe
**df.write.format("delta").saveAsTable("main.demo_schema.scd2_table_demo")**

In [0]:
# Describe the table
spark.sql("DESCRIBE  main.demo_schema.scd2_table_demo").display()

col_name,data_type,comment
new_hash,string,null
id,int,null
name,string,null
price,double,null
effective_date,timestamp,null
end_date,timestamp,null


In [0]:
# Extended describe
spark.sql("DESCRIBE EXTENDED main.demo_schema.scd2_table_demo").display()

col_name,data_type,comment
new_hash,string,null
id,int,null
name,string,null
price,double,null
effective_date,timestamp,null
end_date,timestamp,null
,,
# Detailed Table Information,,
Catalog,main,
Database,demo_schema,


In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

new_hash,id,name,price,effective_date,end_date


In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import lit,current_timestamp
from pyspark.sql.types import StringType,TimestampType,IntegerType,DoubleType
dt= DeltaTable.forName(spark,"main.demo_schema.scd2_table_demo")


In [0]:
# update Current row
dt.alias("t").\
    merge(
        source=initial_df.alias("s"),
        condition="s.new_hash=t.new_hash and t.end_date is null"
    ).\
        whenMatchedUpdate(
            set ={"t.end_date":current_timestamp()}
                          ).\
                        execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

new_hash,id,name,price,effective_date,end_date


In [0]:
initial_df=initial_df.\
    withColumn("effective_date",current_timestamp()).\
    withColumn("end_date",lit(None).cast(TimestampType())).\
    select(initial_df.new_hash,initial_df.id.cast(IntegerType()),initial_df.name.cast(StringType()),initial_df.price.cast(DoubleType()),"effective_date","end_date").\
    write.\
    mode("append").\
    saveAsTable("main.demo_schema.scd2_table_demo")


In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

new_hash,id,name,price,effective_date,end_date
bc4ca3e7265ea6648b12cb219796a29fb57490b724e2e35573c58ecf3e2a8966,1,Widget A,10.0,2026-01-13T09:26:18.828Z,null
b2f924f33af193f4bdd5732036f7e911556c30dceb3a5f5527e859de7b416e84,2,Widget B,12.5,2026-01-13T09:26:18.828Z,null
021b016051b782d0982dec293c01002a8e94d42a9ff1fb683932ce3277b86d7b,3,Widget C,20.0,2026-01-13T09:26:18.828Z,null


In [0]:
# Sample initial data
initial_upd = spark.createDataFrame(
    [
     (2, "Widget B",15)],
    ["id", "name", "price"]
)
initial_upd=initial_upd.withColumn("new_hash",sha2(concat_ws(lit('_'),col("id"),col("name")),256))
initial_upd.show()

+---+--------+-----+--------------------+
| id|    name|price|            new_hash|
+---+--------+-----+--------------------+
|  2|Widget B|   15|b2f924f33af193f4b...|
+---+--------+-----+--------------------+



In [0]:
# update Current row
dt.alias("t").\
    merge(
        source=initial_upd.alias("s"),
        condition="s.new_hash=t.new_hash and t.end_date is null"
    ).\
        whenMatchedUpdate(
            set ={"t.end_date":current_timestamp()}
                          ).\
                        execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

new_hash,id,name,price,effective_date,end_date
b2f924f33af193f4bdd5732036f7e911556c30dceb3a5f5527e859de7b416e84,2,Widget B,12.5,2026-01-13T09:26:18.828Z,2026-01-13T09:26:21.745Z
bc4ca3e7265ea6648b12cb219796a29fb57490b724e2e35573c58ecf3e2a8966,1,Widget A,10.0,2026-01-13T09:26:18.828Z,null
021b016051b782d0982dec293c01002a8e94d42a9ff1fb683932ce3277b86d7b,3,Widget C,20.0,2026-01-13T09:26:18.828Z,null


In [0]:
initial_upd=initial_upd.withColumn("effective_date",current_timestamp()).withColumn("end_date",lit(None).cast(TimestampType())).select(initial_upd.new_hash,initial_upd.id.cast(IntegerType()),initial_upd.name.cast(StringType()),initial_upd.price.cast(DoubleType()),"effective_date","end_date").write.mode("append").saveAsTable("main.demo_schema.scd2_table_demo")


In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

new_hash,id,name,price,effective_date,end_date
bc4ca3e7265ea6648b12cb219796a29fb57490b724e2e35573c58ecf3e2a8966,1,Widget A,10.0,2026-01-13T09:26:18.828Z,null
021b016051b782d0982dec293c01002a8e94d42a9ff1fb683932ce3277b86d7b,3,Widget C,20.0,2026-01-13T09:26:18.828Z,null
b2f924f33af193f4bdd5732036f7e911556c30dceb3a5f5527e859de7b416e84,2,Widget B,12.5,2026-01-13T09:26:18.828Z,2026-01-13T09:26:21.745Z
b2f924f33af193f4bdd5732036f7e911556c30dceb3a5f5527e859de7b416e84,2,Widget B,15.0,2026-01-13T09:26:27.746Z,null


In [0]:
# Sample initial data
initial_upd1 = spark.createDataFrame(
    [
     (1, "Widget BA",15)],
    ["id", "name", "price"]
)
initial_upd1=initial_upd1.withColumn("new_hash",sha2(concat_ws(lit('_'),col("id"),col("name")),256))
initial_upd1.show()

+---+---------+-----+--------------------+
| id|     name|price|            new_hash|
+---+---------+-----+--------------------+
|  1|Widget BA|   15|d0cb3ac238fbcfc2e...|
+---+---------+-----+--------------------+



In [0]:
# update Current row
dt.alias("t").\
    merge(
        source=initial_upd1.alias("s"),
        condition="s.new_hash=t.new_hash and t.end_date is null"
    ).\
        whenMatchedUpdate(
            set ={"t.end_date":current_timestamp()}
                          ).\
        whenNotMatchedInsert(
            values={
             "t.new_hash": "s.new_hash",
             "t.id": "s.id",
             "t.name": "s.name",
             "t.price": "s.price",
             "t.effective_date": current_timestamp(),
             "t.end_date": lit(None).cast(TimestampType())
        }
            ).\
                        execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

new_hash,id,name,price,effective_date,end_date
bc4ca3e7265ea6648b12cb219796a29fb57490b724e2e35573c58ecf3e2a8966,1,Widget A,10.0,2026-01-13T09:26:18.828Z,null
021b016051b782d0982dec293c01002a8e94d42a9ff1fb683932ce3277b86d7b,3,Widget C,20.0,2026-01-13T09:26:18.828Z,null
b2f924f33af193f4bdd5732036f7e911556c30dceb3a5f5527e859de7b416e84,2,Widget B,12.5,2026-01-13T09:26:18.828Z,2026-01-13T09:26:21.745Z
d0cb3ac238fbcfc2ef53e938e1421798b29db926c0b0781343f6d6971017aea8,1,Widget BA,15.0,2026-01-13T09:26:30.446Z,null
b2f924f33af193f4bdd5732036f7e911556c30dceb3a5f5527e859de7b416e84,2,Widget B,15.0,2026-01-13T09:26:27.746Z,null


In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").explain()

== Physical Plan ==
*(1) ColumnarToRow
+- PhotonResultStage
   +- PhotonScan parquet main.demo_schema.scd2_table_demo[new_hash#18168,id#18169,name#18170,price#18171,effective_date#18172,end_date#18173] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-lw6mu/uc/f8d04c04-4088-4d2a-9975-0080fdf78f74..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<new_hash:string,id:int,name:string,price:double,effective_date:timestamp,end_date:timestamp>, RequiredDataFilters: []


== Photon Explanation ==
The query is fully supported by Photon.
== Optimizer Statistics (table names per statistics state) ==
  missing = 
  partial = 
  full    = scd2_table_demo



In [0]:
spark.sql("alter table main.demo_schema.scd2_table_demo set TBLPROPERTIES( delta.enableChangeDataFeed=True)")

DataFrame[]

In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

new_hash,id,name,price,effective_date,end_date
bc4ca3e7265ea6648b12cb219796a29fb57490b724e2e35573c58ecf3e2a8966,1,Widget A,10.0,2026-01-13T09:26:18.828Z,null
021b016051b782d0982dec293c01002a8e94d42a9ff1fb683932ce3277b86d7b,3,Widget C,20.0,2026-01-13T09:26:18.828Z,null
b2f924f33af193f4bdd5732036f7e911556c30dceb3a5f5527e859de7b416e84,2,Widget B,12.5,2026-01-13T09:26:18.828Z,2026-01-13T09:26:21.745Z
d0cb3ac238fbcfc2ef53e938e1421798b29db926c0b0781343f6d6971017aea8,1,Widget BA,15.0,2026-01-13T09:26:30.446Z,null
b2f924f33af193f4bdd5732036f7e911556c30dceb3a5f5527e859de7b416e84,2,Widget B,15.0,2026-01-13T09:26:27.746Z,null


In [0]:
# Sample initial data
initial_del1 = spark.createDataFrame(
    [
     (1, "Widget BA",15)],
    ["id", "name", "price"]
)
initial_del1=initial_del1.withColumn("new_hash",sha2(concat_ws(lit('_'),col("id"),col("name")),256))
initial_del1.show()

+---+---------+-----+--------------------+
| id|     name|price|            new_hash|
+---+---------+-----+--------------------+
|  1|Widget BA|   15|d0cb3ac238fbcfc2e...|
+---+---------+-----+--------------------+



In [0]:
# delete   row
dt.alias("t").\
    merge(
        source=initial_del1.alias("s"),
        condition="s.new_hash=t.new_hash"
    ).\
    whenMatchedDelete().\
                        execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

new_hash,id,name,price,effective_date,end_date
bc4ca3e7265ea6648b12cb219796a29fb57490b724e2e35573c58ecf3e2a8966,1,Widget A,10.0,2026-01-13T09:26:18.828Z,null
021b016051b782d0982dec293c01002a8e94d42a9ff1fb683932ce3277b86d7b,3,Widget C,20.0,2026-01-13T09:26:18.828Z,null
b2f924f33af193f4bdd5732036f7e911556c30dceb3a5f5527e859de7b416e84,2,Widget B,12.5,2026-01-13T09:26:18.828Z,2026-01-13T09:26:21.745Z
b2f924f33af193f4bdd5732036f7e911556c30dceb3a5f5527e859de7b416e84,2,Widget B,15.0,2026-01-13T09:26:27.746Z,null
